In [3]:

from google.colab import drive
from pathlib import Path

MOUNT = Path("/content/drive5")

# Refuse to overwrite an existing non-empty directory.
if MOUNT.exists() and any(MOUNT.iterdir()):
    raise RuntimeError(
        "/content/drive5 is not empty. Choose another unused mount path."
    )

drive.mount(str(MOUNT))

PROJECT = MOUNT / "MyDrive/Ocular_Project"

print("\nProject exists:", PROJECT.exists())

if PROJECT.exists():
    print("Project contents:")
    for p in PROJECT.iterdir():
        print(("DIR " if p.is_dir() else "FILE "), p.name)

for relative in [
    "datasets/IITD_database (1).7z",
    "results/IITD/notebook01_dataset_analysis/IITD_dataset_metadata.csv",
    "results/IITD/notebook02_preprocessing/IITD_preprocessed_metadata.csv",
]:
    p = PROJECT / relative
    print(f"\n{relative}: {p.exists()}")


Mounted at /content/drive5

Project exists: True
Project contents:
DIR  datasets
DIR  models
DIR  generated_images
DIR  results

datasets/IITD_database (1).7z: True

results/IITD/notebook01_dataset_analysis/IITD_dataset_metadata.csv: True

results/IITD/notebook02_preprocessing/IITD_preprocessed_metadata.csv: True


In [5]:

print("Original metadata columns:")
print(original_df.columns.tolist())

print("\nPreprocessed metadata columns:")
print(processed_df.columns.tolist())

print("\nOriginal metadata sample:")
display(original_df.head(2))

print("\nPreprocessed metadata sample:")
display(processed_df.head(2))


Original metadata columns:
['image_path', 'relative_path', 'filename', 'subject', 'eye', 'capture_id', 'extension', 'valid_image', 'width', 'height', 'mode', 'format', 'file_size_bytes']

Preprocessed metadata columns:
['original_relative_path', 'processed_path', 'subject', 'eye', 'capture_id', 'width', 'height', 'mode']

Original metadata sample:


,image_path,relative_path,filename,subject,eye,capture_id,extension,valid_image,width,height,mode,format,file_size_bytes
0,/content/drive/MyDrive/Ocular_Project/datasets...,001/01_L.bmp,01_L.bmp,001,L,1,.bmp,True,320,240,RGB,BMP,230454
1,/content/drive/MyDrive/Ocular_Project/datasets...,001/02_L.bmp,02_L.bmp,001,L,2,.bmp,True,320,240,RGB,BMP,230454



Preprocessed metadata sample:


,original_relative_path,processed_path,subject,eye,capture_id,width,height,mode
0,001/01_L.bmp,/content/gdrive/MyDrive/Ocular_Project/results...,001,L,1,224,224,L
1,001/02_L.bmp,/content/gdrive/MyDrive/Ocular_Project/results...,001,L,2,224,224,L


In [6]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive5/MyDrive/Ocular_Project")

ORIGINAL_META = (
    PROJECT / "results/IITD/notebook01_dataset_analysis/"
    "IITD_dataset_metadata.csv"
)
PREPROCESSED_META = (
    PROJECT / "results/IITD/notebook02_preprocessing/"
    "IITD_preprocessed_metadata.csv"
)

original_df = pd.read_csv(ORIGINAL_META)
processed_df = pd.read_csv(PREPROCESSED_META)

# Select original 320x240 images only.
canonical = original_df[
    (original_df["width"] == 320) &
    (original_df["height"] == 240) &
    (original_df["valid_image"].astype(str).str.lower() == "true")
].copy()

# Correct merge: the two files use different path column names.
df = canonical.merge(
    processed_df,
    left_on="relative_path",
    right_on="original_relative_path",
    how="inner",
    suffixes=("_original", "_processed")
)

# Correct paths from older Drive mounts.
for old_mount in [
    "/content/gdrive/MyDrive/",
    "/content/gdrive2/MyDrive/",
    "/content/drive/MyDrive/",
    "/content/drive2/MyDrive/",
]:
    df["processed_path"] = df["processed_path"].str.replace(
        old_mount,
        "/content/drive5/MyDrive/",
        regex=False
    )

df["path_exists"] = df["processed_path"].map(
    lambda p: Path(p).is_file()
)

print("Original metadata rows:", len(original_df))
print("Preprocessed metadata rows:", len(processed_df))
print("Canonical images matched:", len(df))
print("Accessible processed images:", int(df["path_exists"].sum()))
print("Missing processed images:", int((~df["path_exists"]).sum()))

print("\nMatched image examples:")
display(df[[
    "relative_path",
    "original_relative_path",
    "processed_path",
    "subject_original",
    "eye_original"
]].head())

if df.empty or not df["path_exists"].all():
    raise FileNotFoundError(
        "Merge produced no rows or some processed images are inaccessible."
    )

print("\nSetup verified. Ready for baseline training.")


Original metadata rows: 3270
Preprocessed metadata rows: 3270
Canonical images matched: 2180
Accessible processed images: 2180
Missing processed images: 0

Matched image examples:


,relative_path,original_relative_path,processed_path,subject_original,eye_original
0,001/01_L.bmp,001/01_L.bmp,/content/drive5/MyDrive/Ocular_Project/results...,001,L
1,001/02_L.bmp,001/02_L.bmp,/content/drive5/MyDrive/Ocular_Project/results...,001,L
2,001/03_L.bmp,001/03_L.bmp,/content/drive5/MyDrive/Ocular_Project/results...,001,L
3,001/04_L.bmp,001/04_L.bmp,/content/drive5/MyDrive/Ocular_Project/results...,001,L
4,001/05_L.bmp,001/05_L.bmp,/content/drive5/MyDrive/Ocular_Project/results...,001,L



Setup verified. Ready for baseline training.


In [8]:

# Inspect class counts in the current baseline dataframe
class_counts = df["label_name"].value_counts().sort_values()

print("Total images:", len(df))
print("Total subject-eye classes:", df["label_name"].nunique())

print("\nImages per class:")
print(class_counts.value_counts().sort_index().to_string())

print("\nClasses with fewer than 3 images:", (class_counts < 3).sum())
print("Classes with fewer than 5 images:", (class_counts < 5).sum())

print("\nSmallest class counts:")
display(class_counts.head(15).rename("image_count").to_frame())


Total images: 2180
Total subject-eye classes: 425

Images per class:
count
4       3
5     408
6       3
10     11

Classes with fewer than 3 images: 0
Classes with fewer than 5 images: 3

Smallest class counts:


,image_count
label_name,
207_R,4
206_R,4
205_R,4
202_R,5
031_L,5
203_L,5
003_R,5
005_R,5
001_L,5


In [9]:

# ============================================================
# NOTEBOOK 04 — BASELINE RECOGNITION MODEL
# IIT Delhi Iris Database | ResNet18 | No augmentation
# ============================================================

import random
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

# 1. Reproducibility and device
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

PROJECT = Path("/content/drive5/MyDrive/Ocular_Project")
OUT_DIR = PROJECT / "results/IITD/notebook04_baseline_recognition"
MODEL_DIR = PROJECT / "models/IITD"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


# 2. Prepare labels
df = df.copy().reset_index(drop=True)
df["subject"] = df["subject_original"].astype(str).str.zfill(3)
df["eye"] = df["eye_original"].astype(str).str.upper()
df["label_name"] = df["subject"] + "_" + df["eye"]

encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["label_name"])

# Single stratified split: 70% train, 15% validation, 15% test.
# This avoids splitting a small temporary set a second time.
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    splitter.split(np.zeros(len(df)), df["label"])
)

# Split the remaining 30% into validation and test.
# Small classes may not have enough samples for stratification here,
# so allocate the temporary samples class-by-class.
temp_df = df.iloc[temp_idx].copy()
val_parts, test_parts = [], []

rng = np.random.default_rng(42)

for label, group in temp_df.groupby("label"):
    group_indices = group.index.to_numpy()
    rng.shuffle(group_indices)

    # Keep at least one validation and one test example where possible.
    if len(group_indices) >= 2:
        n_val = max(1, len(group_indices) // 2)
        n_val = min(n_val, len(group_indices) - 1)
        val_parts.extend(group_indices[:n_val])
        test_parts.extend(group_indices[n_val:])
    else:
        # A singleton in the temporary set goes to test.
        test_parts.extend(group_indices)

val_idx = np.asarray(val_parts, dtype=int)
test_idx = np.asarray(test_parts, dtype=int)

# Save exact split assignments.
df["split"] = "unused"
df.loc[train_idx, "split"] = "train"
df.loc[val_idx, "split"] = "validation"
df.loc[test_idx, "split"] = "test"

df.to_csv(OUT_DIR / "IITD_baseline_dataset_split.csv", index=False)

print("Images:", len(df))
print("Classes:", df["label"].nunique())
print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))
print("\nSplit counts:")
print(df["split"].value_counts())
print("\nClasses absent from training:",
      len(set(df["label"]) - set(df.iloc[train_idx]["label"])))


# 3. Dataset
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

class IITDDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(row["processed_path"]) as im:
            image = transform(im.convert("L"))
        return image, int(row["label"])

dataset = IITDDataset(df)

BATCH_SIZE = 16 if DEVICE.type == "cpu" else 32
NUM_WORKERS = 0  # reliable on Colab and avoids worker overhead

train_loader = DataLoader(
    Subset(dataset, train_idx.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)
val_loader = DataLoader(
    Subset(dataset, val_idx.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)
test_loader = DataLoader(
    Subset(dataset, test_idx.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

# 4. ResNet18 baseline model
NUM_CLASSES = len(encoder.classes_)

model = models.resnet18(weights=None)
model.conv1 = nn.Conv2d(
    1, 64, kernel_size=7, stride=2, padding=3, bias=False
)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 5
best_val_accuracy = -1.0
history = []

# 5. Train and validate
for epoch in range(EPOCHS):
    start = time.time()
    model.train()
    loss_sum = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(
        train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False
    ):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss = loss_sum / total
    train_acc = correct / total

    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss_sum += loss.item() * labels.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "validation_loss": val_loss,
        "validation_accuracy": val_acc,
        "seconds": time.time() - start,
    })

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train loss {train_loss:.4f} | "
        f"Train acc {train_acc:.2%} | "
        f"Val loss {val_loss:.4f} | "
        f"Val acc {val_acc:.2%} | "
        f"{history[-1]['seconds']:.1f}s"
    )

    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "num_classes": NUM_CLASSES,
            "class_names": encoder.classes_.tolist(),
            "seed": SEED,
        }, MODEL_DIR / "IITD_baseline_resnet18_best.pth")

# 6. Save history
pd.DataFrame(history).to_csv(
    OUT_DIR / "IITD_baseline_training_history.csv", index=False
)

# 7. Evaluate the best checkpoint on test data
checkpoint_path = MODEL_DIR / "IITD_baseline_resnet18_best.pth"
checkpoint = torch.load(
    checkpoint_path, map_location=DEVICE, weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        outputs = model(images.to(DEVICE))
        y_pred.extend(outputs.argmax(1).cpu().numpy())
        y_true.extend(labels.numpy())

metrics = {
    "test_images": len(y_true),
    "num_classes": NUM_CLASSES,
    "accuracy": accuracy_score(y_true, y_pred),
    "macro_precision": precision_score(
        y_true, y_pred, average="macro", zero_division=0
    ),
    "macro_recall": recall_score(
        y_true, y_pred, average="macro", zero_division=0
    ),
    "macro_f1": f1_score(
        y_true, y_pred, average="macro", zero_division=0
    ),
    "best_validation_accuracy": best_val_accuracy,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "architecture": "ResNet18",
    "pretrained": False,
    "augmentation": "None",
}

with open(OUT_DIR / "IITD_baseline_test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

pd.DataFrame([metrics]).to_csv(
    OUT_DIR / "IITD_baseline_test_metrics.csv", index=False
)

report = classification_report(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    target_names=encoder.classes_,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(report).transpose().to_csv(
    OUT_DIR / "IITD_baseline_classification_report.csv"
)

print("\n===== BASELINE TEST RESULTS =====")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float)
          else f"{key}: {value}")

print("\nModel saved:", checkpoint_path)
print("Results saved:", OUT_DIR)
print("Training completed.")


Device: cpu
Images: 2180
Classes: 425
Train: 1526
Validation: 218
Test: 436

Split counts:
split
train         1526
test           436
validation     218
Name: count, dtype: int64

Classes absent from training: 0


Epoch 1/5:   0%|          | 0/96 [00:00<?, ?it/s]

Epoch 1/5 | Train loss 6.5178 | Train acc 0.46% | Val loss 7.3266 | Val acc 0.92% | 924.9s


Epoch 2/5:   0%|          | 0/96 [00:00<?, ?it/s]

Epoch 2/5 | Train loss 5.8001 | Train acc 1.44% | Val loss 5.4326 | Val acc 2.75% | 451.2s


...

Epoch 4/5 | Train loss 3.7925 | Train acc 14.09% | Val loss 3.4815 | Val acc 15.60% | 443.3s


Epoch 5/5:   0%|          | 0/96 [00:00<?, ?it/s]

Epoch 5/5 | Train loss 2.7450 | Train acc 29.29% | Val loss 2.3521 | Val acc 41.28% | 458.5s


Testing:   0%|          | 0/28 [00:00<?, ?it/s]


===== BASELINE TEST RESULTS =====
test_images: 436
num_classes: 425
accuracy: 0.4725
macro_precision: 0.3414
macro_recall: 0.4647
macro_f1: 0.3718
best_validation_accuracy: 0.4128
epochs: 5
batch_size: 16
architecture: ResNet18
pretrained: False
augmentation: None

Model saved: /content/drive5/MyDrive/Ocular_Project/models/IITD/IITD_baseline_resnet18_best.pth
Results saved: /content/drive5/MyDrive/Ocular_Project/results/IITD/notebook04_baseline_recognition
Training completed.


## Conclusion

In this notebook, a baseline ocular recognition model was developed and evaluated using the IIT Delhi Iris Database. The experiment used **2,180 canonical images** with a resolution of 224 × 224 pixels in grayscale format. The left and right eyes of each subject were treated as separate recognition classes, resulting in **425 subject-eye classes**.

A ResNet18 convolutional neural network was trained from scratch for five epochs using the Adam optimizer, a learning rate of 0.001, a batch size of 16, and cross-entropy loss. No synthetic images or data augmentation were used, establishing a baseline for subsequent experiments.

### Dataset Split

| Split | Number of Images |
|---|---:|
| Training | 1,526 |
| Validation | 218 |
| Testing | 436 |
| **Total** | **2,180** |

### Test Results

| Evaluation Metric | Result |
|---|---:|
| Accuracy | 47.25% |
| Macro Precision | 34.14% |
| Macro Recall | 46.47% |
| Macro F1-score | 37.18% |
| Best Validation Accuracy | 41.28% |

### Training Observations

Training accuracy increased from 0.46% in the first epoch to 29.29% in the fifth epoch. Validation accuracy also improved, reaching 41.28% in the final epoch. These trends indicate that the model learned useful discriminative patterns during training, although additional experiments would be needed to determine whether further training or other model configurations could improve performance.

The model achieved 47.25% accuracy on the held-out test set. The difference between accuracy and macro-averaged precision and F1-score highlights the importance of considering multiple metrics when evaluating recognition performance across numerous classes.

### Generated Outputs

The model checkpoint and evaluation artifacts were saved for reproducibility and future comparisons:

- `IITD_baseline_resnet18_best.pth` — best model checkpoint selected using validation accuracy.
- `IITD_baseline_dataset_split.csv` — saved dataset split assignments.
- `IITD_baseline_training_history.csv` — epoch-wise training and validation metrics.
- `IITD_baseline_test_metrics.csv` — final test metrics in CSV format.
- `IITD_baseline_test_metrics.json` — final test metrics in JSON format.
- `IITD_baseline_classification_report.csv` — per-class classification report.

### Limitations

The experiment used an image-level split, meaning images from the same subject-eye class can appear in training, validation, and testing. Consequently, these results measure closed-set recognition performance rather than generalization to entirely unseen identities.

Training was performed on a CPU for five epochs. Further experiments with optimized data loading, additional training, and controlled comparisons may improve performance and provide a clearer understanding of the model's capabilities.

### Next Steps

This baseline provides a reference point for evaluating subsequent augmentation strategies. Future experiments should reuse the same dataset split and evaluation protocol wherever possible, allowing differences in recognition performance to be interpreted more reliably.